# ScholarRAG V2 — Complete End-to-End Colab Notebook
**All 8 phases in one notebook.** Run from top to bottom.

1. PDF ingestion & chunking
2. Embeddings & SQLite storage
3. Semantic retrieval
4. Hybrid retrieval (semantic + BM25)
5. Reranking
6. Grounded answer generation (Groq optional; extractive fallback)
7. Evaluation & diagnostics
8. Gradio UI

**Persistence:** Colab `/content` is temporary. Optional Google Drive mount is provided.
**Verification:** Code syntax is checked here; live Colab/model execution must be verified by running the notebook.

## 0 — Setup

In [1]:
!pip -q install pymupdf sentence-transformers rank-bm25 gradio groq

In [2]:
import os, re, json, sqlite3, hashlib, warnings, textwrap, time
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Any, Optional
import numpy as np
import fitz
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
print("Setup imports complete.")

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Setup imports complete.


### Optional persistent Google Drive storage

In [3]:
USE_GOOGLE_DRIVE = False  # Change to True to mount Drive and persist DB between sessions.
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/ScholarRAG_V2')
else:
    DATA_DIR = Path('/content/ScholarRAG_V2')
DATA_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = DATA_DIR / 'scholarrag.sqlite3'
print("Database:", DB_PATH)

Database: /content/ScholarRAG_V2/scholarrag.sqlite3


## Phase 1 — PDF ingestion, cleaning, and chunking

In [4]:
@dataclass
class Document:
    document_id: str
    filename: str
    sha256: str
    pages: int
    extracted_characters: int
    warnings: List[str]

@dataclass
class Chunk:
    chunk_id: str
    document_id: str
    filename: str
    text: str
    page_start: int
    page_end: int
    chunk_index: int
    word_count: int

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def normalize_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def extract_pdf(path: Path):
    pdf = fitz.open(path)
    page_texts, low_text_pages = [], []
    for i, page in enumerate(pdf, start=1):
        clean = normalize_text(page.get_text("text") or "")
        page_texts.append(clean)
        if len(clean) < 40:
            low_text_pages.append(i)
    full_text = "\n\n".join(page_texts)
    file_hash = sha256_file(path)
    warnings_list = []
    if low_text_pages:
        warnings_list.append(
            f"{len(low_text_pages)}/{len(pdf)} pages have little extracted text; OCR may be needed. "
            f"Low-text pages (first 30): {low_text_pages[:30]}"
        )
    doc = Document(
        document_id=file_hash[:32], filename=path.name, sha256=file_hash,
        pages=len(pdf), extracted_characters=len(full_text), warnings=warnings_list
    )
    pdf.close()
    return doc, page_texts

def chunk_pages(document: Document, page_texts: List[str],
                target_words: int = 500, overlap_words: int = 80) -> List[Chunk]:
    tokens = []
    for page_no, text in enumerate(page_texts, start=1):
        tokens.extend((word, page_no) for word in text.split())
    chunks, start, idx = [], 0, 0
    step = max(1, target_words - overlap_words)
    while start < len(tokens):
        end = min(start + target_words, len(tokens))
        selected = tokens[start:end]
        chunk_text = " ".join(w for w, _ in selected).strip()
        if chunk_text:
            pages = [p for _, p in selected]
            cid = hashlib.sha256(f"{document.document_id}:{idx}:{chunk_text}".encode()).hexdigest()[:32]
            chunks.append(Chunk(cid, document.document_id, document.filename, chunk_text,
                                min(pages), max(pages), idx, len(selected)))
        if end >= len(tokens):
            break
        start += step
        idx += 1
    return chunks

def ingest_pdf(path: Path):
    doc, page_texts = extract_pdf(path)
    return doc, chunk_pages(doc, page_texts)

try:
    from google.colab import files
    uploaded = files.upload()
    PDF_PATHS = [Path(name) for name in uploaded if name.lower().endswith(".pdf")]
except Exception:
    PDF_PATHS = []
    print("Not in Colab: set PDF_PATHS manually.")

documents, chunks_by_document = [], {}
for pdf_path in PDF_PATHS:
    print("\nIngesting:", pdf_path.name)
    doc, doc_chunks = ingest_pdf(pdf_path)
    documents.append(doc)
    chunks_by_document[doc.document_id] = doc_chunks
    print({"filename": doc.filename, "pages": doc.pages, "characters": doc.extracted_characters,
           "chunks": len(doc_chunks), "warnings": doc.warnings})

all_chunks = [c for d in documents for c in chunks_by_document[d.document_id]]
assert documents and all_chunks, "Upload at least one readable PDF and ensure it generated chunks."
print(f"Phase 1 complete: {len(documents)} PDFs, {len(all_chunks)} chunks.")

Saving textbook_full.pdf to textbook_full.pdf

Ingesting: textbook_full.pdf
{'filename': 'textbook_full.pdf', 'pages': 187, 'characters': 300037, 'chunks': 125, 'warnings': ['13/187 pages have little extracted text; OCR may be needed. Low-text pages (first 30): [30, 36, 38, 46, 98, 107, 108, 110, 111, 116, 118, 158, 172]']}
Phase 1 complete: 1 PDFs, 125 chunks.


## Phase 2 — Embeddings and persistent SQLite storage

In [5]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIM = 384
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

def embed_texts(texts: List[str], batch_size: int = 32) -> np.ndarray:
    if not texts:
        return np.empty((0, EMBEDDING_DIM), dtype=np.float32)
    arr = embedding_model.encode(texts, batch_size=batch_size, show_progress_bar=True,
                                convert_to_numpy=True, normalize_embeddings=True)
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim != 2 or arr.shape[1] != EMBEDDING_DIM:
        raise ValueError(f"Unexpected embedding shape: {arr.shape}")
    return arr

def connect_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("PRAGMA foreign_keys = ON")
    return conn

def init_db():
    with connect_db() as conn:
        conn.executescript("""
        CREATE TABLE IF NOT EXISTS documents (
            document_id TEXT PRIMARY KEY, filename TEXT NOT NULL, sha256 TEXT NOT NULL UNIQUE,
            pages INTEGER, extracted_characters INTEGER
        );
        CREATE TABLE IF NOT EXISTS chunks (
            chunk_id TEXT PRIMARY KEY, document_id TEXT NOT NULL REFERENCES documents(document_id) ON DELETE CASCADE,
            filename TEXT NOT NULL, text TEXT NOT NULL, page_start INTEGER, page_end INTEGER,
            chunk_index INTEGER, word_count INTEGER
        );
        CREATE TABLE IF NOT EXISTS embeddings (
            chunk_id TEXT PRIMARY KEY REFERENCES chunks(chunk_id) ON DELETE CASCADE,
            dimension INTEGER NOT NULL, vector BLOB NOT NULL
        );
        """)
init_db()

def save_document(doc: Document, doc_chunks: List[Chunk], vectors: np.ndarray):
    if len(doc_chunks) != len(vectors):
        raise ValueError("Chunk/vector count mismatch.")
    with connect_db() as conn:
        old = conn.execute("SELECT document_id FROM documents WHERE sha256=?", (doc.sha256,)).fetchone()
        if old:
            conn.execute("DELETE FROM documents WHERE document_id=?", (old[0],))
        conn.execute("INSERT INTO documents VALUES (?, ?, ?, ?, ?)",
                     (doc.document_id, doc.filename, doc.sha256, doc.pages, doc.extracted_characters))
        for chunk, vector in zip(doc_chunks, vectors):
            conn.execute("INSERT INTO chunks VALUES (?, ?, ?, ?, ?, ?, ?, ?)",
                         (chunk.chunk_id, chunk.document_id, chunk.filename, chunk.text,
                          chunk.page_start, chunk.page_end, chunk.chunk_index, chunk.word_count))
            conn.execute("INSERT INTO embeddings VALUES (?, ?, ?)",
                         (chunk.chunk_id, EMBEDDING_DIM, np.asarray(vector, dtype=np.float32).tobytes()))
    return {"filename": doc.filename, "chunks_saved": len(doc_chunks)}

def load_chunks_and_vectors():
    with connect_db() as conn:
        rows = conn.execute("""
            SELECT c.chunk_id, c.document_id, c.filename, c.text, c.page_start, c.page_end,
                   c.chunk_index, c.word_count, e.dimension, e.vector
            FROM chunks c JOIN embeddings e ON e.chunk_id=c.chunk_id
            ORDER BY c.filename, c.chunk_index
        """).fetchall()
    records, vecs = [], []
    for row in rows:
        records.append({"chunk_id": row[0], "document_id": row[1], "filename": row[2],
                       "text": row[3], "page_start": row[4], "page_end": row[5],
                       "chunk_index": row[6], "word_count": row[7], "dimension": row[8]})
        v = np.frombuffer(row[9], dtype=np.float32)
        if v.size != row[8]:
            raise ValueError(f"Corrupt embedding for chunk {row[0]}")
        vecs.append(v.copy())
    matrix = np.vstack(vecs) if vecs else np.empty((0, EMBEDDING_DIM), dtype=np.float32)
    return records, matrix

def storage_stats():
    with connect_db() as conn:
        n_docs = conn.execute("SELECT COUNT(*) FROM documents").fetchone()[0]
        n_chunks = conn.execute("SELECT COUNT(*) FROM chunks").fetchone()[0]
        dims = [x[0] for x in conn.execute("SELECT DISTINCT dimension FROM embeddings").fetchall()]
        total_bytes = conn.execute("SELECT COALESCE(SUM(LENGTH(vector)),0) FROM embeddings").fetchone()[0]
    return {"documents": n_docs, "chunks": n_chunks, "embedding_dimensions": dims,
            "embedding_bytes": total_bytes}

vectors_by_document, storage_results = {}, []
for doc in documents:
    doc_chunks = chunks_by_document[doc.document_id]
    doc_vectors = embed_texts([c.text for c in doc_chunks])
    vectors_by_document[doc.document_id] = doc_vectors
    storage_results.append(save_document(doc, doc_chunks, doc_vectors))
all_vectors = np.vstack([vectors_by_document[d.document_id] for d in documents]).astype(np.float32)
loaded_records, loaded_vectors = load_chunks_and_vectors()
print("Storage results:", storage_results)
print("Storage stats:", storage_stats())
print("Phase 2 completed.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Storage results: [{'filename': 'textbook_full.pdf', 'chunks_saved': 125}]
Storage stats: {'documents': 1, 'chunks': 125, 'embedding_dimensions': [384], 'embedding_bytes': 192000}
Phase 2 completed.


## Phase 3 — Semantic retrieval

In [6]:
retrieval_records, retrieval_vectors = load_chunks_and_vectors()
assert retrieval_vectors.shape[0] == len(retrieval_records)

def semantic_search(query: str, top_k: int = 5):
    qvec = embed_texts([query])[0]
    scores = retrieval_vectors @ qvec
    order = np.argsort(scores)[::-1][:min(top_k, len(scores))]
    return [{**retrieval_records[i], "semantic_score": float(scores[i])} for i in order]

TEST_QUERY = "What are the main concepts discussed in this textbook?"
semantic_results = semantic_search(TEST_QUERY, top_k=5)
for rank, hit in enumerate(semantic_results, start=1):
    print(f"\nRank {rank} | score={hit['semantic_score']:.4f} | {hit['filename']} | "
          f"pages {hit['page_start']}-{hit['page_end']}")
    print(hit["text"][:650], "...")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Rank 1 | score=0.4204 | textbook_full.pdf | pages 1-3
1 Introduction Introduction to Artificial Intelligence By Nikhil Sharma, Josh Hug, Jacky Liang, and Henry Zhu. Additional Contributions by Regina Wang, Peyrin Kao, Matei Gardea, Danial Toktarbayev, Catherine Chu, Samantha Huang, Saathvik Selvan, Pranav Muralikrishnan, and Wesley Zheng. Website Established By Wesley Zheng. Special Thanks to Ashley Chiu. This is the textbook for CS 188: Introduction to Artificial Intelligence at UC Berkeley. It provides an introduction to the full range of topics studied in artificial intelligence, with emphasis on the “core competences” of intelligent systems - problem solving, reasoning, decision making, and  ...

Rank 2 | score=0.2696 | textbook_full.pdf | pages 48-48
that state. We’ll formalize the concept of utility more concretely later, but for now it’s enough to simply think of an agent’s utility as its score or number of points it attains. The value of a terminal state, called a terminal uti

## Phase 4 — Hybrid retrieval (vector + BM25)

In [7]:
def tokenize_for_bm25(text):
    return re.findall(r"[a-zA-Z0-9_]+", text.lower())

bm25_corpus = [tokenize_for_bm25(r["text"]) for r in retrieval_records]
bm25 = BM25Okapi(bm25_corpus) if bm25_corpus else None

def hybrid_search(query, top_k=10, semantic_k=20, keyword_k=20):
    sem_hits = semantic_search(query, top_k=min(semantic_k, len(retrieval_records)))
    bm_scores = bm25.get_scores(tokenize_for_bm25(query)) if bm25 else np.zeros(len(retrieval_records))
    kw_order = np.argsort(bm_scores)[::-1][:min(keyword_k, len(retrieval_records))]
    fused = {}
    for rank, hit in enumerate(sem_hits, 1):
        fused[hit["chunk_id"]] = fused.get(hit["chunk_id"], 0) + 1/(60+rank)
    for rank, idx in enumerate(kw_order, 1):
        cid = retrieval_records[idx]["chunk_id"]
        fused[cid] = fused.get(cid, 0) + 1/(60+rank)
    lookup = {r["chunk_id"]: r for r in retrieval_records}
    ids = sorted(fused, key=fused.get, reverse=True)[:top_k]
    return [{**lookup[cid], "hybrid_score": float(fused[cid])} for cid in ids]

hybrid_results = hybrid_search(TEST_QUERY, top_k=8)
for rank, hit in enumerate(hybrid_results, 1):
    print(f"{rank}. RRF={hit['hybrid_score']:.5f} | {hit['filename']} | "
          f"pages {hit['page_start']}-{hit['page_end']}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

1. RRF=0.03178 | textbook_full.pdf | pages 1-3
2. RRF=0.03078 | textbook_full.pdf | pages 4-5
3. RRF=0.02652 | textbook_full.pdf | pages 168-170
4. RRF=0.02601 | textbook_full.pdf | pages 120-122
5. RRF=0.01639 | textbook_full.pdf | pages 22-25
6. RRF=0.01613 | textbook_full.pdf | pages 48-48
7. RRF=0.01613 | textbook_full.pdf | pages 89-91
8. RRF=0.01587 | textbook_full.pdf | pages 68-70


## Phase 5 — Reranking

In [8]:
RERANK_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANK_MODEL_NAME)

def rerank_results(query, candidates, top_k=5):
    if not candidates:
        return []
    scores = reranker.predict([(query, x["text"]) for x in candidates], show_progress_bar=False)
    out = [{**x, "rerank_score": float(s)} for x, s in zip(candidates, scores)]
    out.sort(key=lambda x: x["rerank_score"], reverse=True)
    return out[:top_k]

reranked_results = rerank_results(TEST_QUERY, hybrid_results, top_k=5)
for rank, hit in enumerate(reranked_results, 1):
    print(f"{rank}. score={hit['rerank_score']:.4f} | {hit['filename']} | "
          f"pages {hit['page_start']}-{hit['page_end']}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

1. score=-3.3297 | textbook_full.pdf | pages 22-25
2. score=-4.4425 | textbook_full.pdf | pages 1-3
3. score=-5.0588 | textbook_full.pdf | pages 168-170
4. score=-5.3446 | textbook_full.pdf | pages 48-48
5. score=-6.0067 | textbook_full.pdf | pages 120-122


## Phase 6 — Grounded answer generation (Groq optional)

In [9]:
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
USE_GROQ = bool(GROQ_API_KEY)
GROQ_MODEL = "llama-3.3-70b-versatile"

def build_context(results, max_chars_per_chunk=1800):
    return "\n\n".join(
        f"[Source: {r['filename']}, pages {r['page_start']}-{r['page_end']}]\n"
        + r["text"][:max_chars_per_chunk] for r in results
    )

def extractive_fallback(query, results):
    if not results:
        return "I couldn't retrieve relevant passages from the indexed PDFs."
    lines = ["No Groq API key configured; showing relevant retrieved excerpts:"]
    for i, r in enumerate(results[:3], 1):
        lines.append(f"\n**Source {i}: {r['filename']} (pages {r['page_start']}-{r['page_end']})**\n"
                     + r["text"][:1200])
    return "\n".join(lines)

def generate_answer(query, results):
    if not USE_GROQ:
        return extractive_fallback(query, results)
    try:
        from groq import Groq
        client = Groq(api_key=GROQ_API_KEY)
        response = client.chat.completions.create(
            model=GROQ_MODEL, temperature=0.1,
            messages=[
                {"role": "system", "content": (
                    "You are ScholarRAG, an academic assistant. Answer only from supplied context. "
                    "If unsupported, say the sources do not establish the answer. Cite filename and page range. "
                    "Do not invent facts or citations."
                )},
                {"role": "user", "content": f"Question: {query}\n\nContext:\n{build_context(results)}"}
            ]
        )
        return response.choices[0].message.content
    except Exception as e:
        print("LLM request failed; falling back to excerpts:", repr(e))
        return extractive_fallback(query, results)

print("Groq enabled:", USE_GROQ)
print(generate_answer(TEST_QUERY, reranked_results))

Groq enabled: False
No Groq API key configured; showing relevant retrieved excerpts:

**Source 1: textbook_full.pdf (pages 22-25)**
the state space and exchanging information between threads. Their main advantage is the use of crossovers — large blocks of letters that have evolved and lead to high valuations can be combined with other such blocks to produce a solution with a high total score. Figure 7: 8-Queens Solution 23 1.6 Summary 1.5 Summary We discussed search problems and their components: a state space, a set of actions, a transition function, an action cost, a start state and a goal state. The agent interacts with the environment through its sensors and its actuators. The agent function describes what the agent does in all circumstances. Rationality of the agent means that the agent seeks to maximize their expected utility. Finally, we define our task environments using PEAS descriptions. Regarding the search problems, they can be solved using a variety of search techniques, i

## Phase 7 — Evaluation & diagnostics

In [10]:
def evaluate_retrieval(query, expected_keywords=None, top_k=5):
    candidates = hybrid_search(query, top_k=10)
    final_hits = rerank_results(query, candidates, top_k=top_k)
    report = {
        "query": query, "candidate_count": len(candidates), "final_count": len(final_hits),
        "top_sources": [{"filename": h["filename"], "pages": [h["page_start"], h["page_end"]]}
                        for h in final_hits]
    }
    if expected_keywords:
        text = " ".join(h["text"].lower() for h in final_hits)
        coverage = {kw: kw.lower() in text for kw in expected_keywords}
        report["keyword_coverage"] = coverage
        report["keyword_coverage_rate"] = sum(coverage.values()) / len(coverage)
    return report

# Customize with textbook questions and keywords that should appear in evidence.
EVALUATION_CASES = [{"question": TEST_QUERY, "expected_keywords": []}]
evaluation_reports = []
for case in EVALUATION_CASES:
    rep = evaluate_retrieval(case["question"], case.get("expected_keywords"))
    evaluation_reports.append(rep)
    print(json.dumps(rep, indent=2))
print("Evaluation diagnostics complete. Keyword coverage is not a full benchmark.")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "query": "What are the main concepts discussed in this textbook?",
  "candidate_count": 10,
  "final_count": 5,
  "top_sources": [
    {
      "filename": "textbook_full.pdf",
      "pages": [
        22,
        25
      ]
    },
    {
      "filename": "textbook_full.pdf",
      "pages": [
        1,
        3
      ]
    },
    {
      "filename": "textbook_full.pdf",
      "pages": [
        168,
        170
      ]
    },
    {
      "filename": "textbook_full.pdf",
      "pages": [
        48,
        48
      ]
    },
    {
      "filename": "textbook_full.pdf",
      "pages": [
        120,
        122
      ]
    }
  ]
}
Evaluation diagnostics complete. Keyword coverage is not a full benchmark.


## Phase 8 — Interactive Gradio UI

In [11]:
import gradio as gr

def scholar_chat(question):
    question = (question or "").strip()
    if not question:
        return "Please enter a question."
    candidates = hybrid_search(question, top_k=10)
    final_hits = rerank_results(question, candidates, top_k=5)
    answer = generate_answer(question, final_hits)
    source_text = "\n\n---\n**Retrieved sources**"
    for i, hit in enumerate(final_hits, 1):
        source_text += (f"\n{i}. **{hit['filename']}**, pages {hit['page_start']}-{hit['page_end']}"
                        f" (rerank score {hit.get('rerank_score', 0):.3f})")
    return answer + source_text

demo = gr.Interface(
    fn=scholar_chat,
    inputs=gr.Textbox(label="Ask your textbook", placeholder="Ask a question grounded in your uploaded PDFs...", lines=2),
    outputs=gr.Markdown(label="ScholarRAG answer"),
    title="ScholarRAG V2",
    description="PDF-grounded academic assistant with semantic + BM25 retrieval and reranking."
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://396f578d5b6ee534de.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Final checklist
- [ ] Phase 1 PDF extraction/chunking completed; investigate low-text pages (OCR is not included).
- [ ] Phase 2 embeddings generated and stored/reloaded.
- [ ] Phase 3 semantic results inspected.
- [ ] Phase 4 hybrid retrieval results inspected.
- [ ] Phase 5 reranking results inspected.
- [ ] Phase 6 grounded answer generation verified (Groq optional).
- [ ] Phase 7 meaningful evaluation questions added.
- [ ] Phase 8 Gradio UI tested.

**Limitations:** OCR is flagged but not automated. Evaluation is a starter diagnostic. SQLite persistence survives Colab restarts only when Google Drive mode is enabled.

In [12]:
# ==========================================
# SCHOLARRAG — GROQ API KEY CONFIGURATION
# ==========================================

import os
from getpass import getpass

GROQ_API_KEY = getpass("Enter your Groq API key (hidden): ")

if not GROQ_API_KEY.strip():
    raise ValueError("Groq API key was not provided.")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY.strip()

print("Groq API key configured successfully! 🔐")

Enter your Groq API key (hidden): ··········
Groq API key configured successfully! 🔐


In [13]:
# ==========================================
# PHASE 6 — GROQ GROUNDED ANSWER GENERATION
# ==========================================

import os
from groq import Groq

# Make sure key was configured
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "").strip()

if not GROQ_API_KEY:
    raise RuntimeError(
        "Groq API key missing. Run the API key setup cell first."
    )

# Groq client
groq_client = Groq(api_key=GROQ_API_KEY)

# Production model ID
GROQ_MODEL = "llama-3.3-70b-versatile"


def build_context(results, max_chars_per_chunk=2500):
    """Prepare retrieved textbook chunks as grounded context."""

    context_parts = []

    for i, result in enumerate(results, start=1):

        source = result.get("filename", "Unknown document")
        page_start = result.get("page_start", "?")
        page_end = result.get("page_end", "?")

        text = result.get("text", "")[:max_chars_per_chunk]

        context_parts.append(
            f"""
[SOURCE {i}]
Document: {source}
Pages: {page_start}-{page_end}

TEXT:
{text}
"""
        )

    return "\n\n".join(context_parts)


def extractive_fallback(query, results):
    """Fallback if the Groq request fails."""

    if not results:
        return "I could not find relevant information in the indexed textbook."

    output = [
        "⚠️ AI generation was unavailable. Here are the retrieved textbook passages:"
    ]

    for i, result in enumerate(results[:3], start=1):

        output.append(
            f"""
### Source {i}: {result.get('filename', 'Unknown')}
**Pages:** {result.get('page_start', '?')}-{result.get('page_end', '?')}

{result.get('text', '')[:1200]}
"""
        )

    return "\n\n".join(output)


def generate_answer(query, results):

    if not results:
        return "I could not retrieve relevant passages from your textbook."

    context = build_context(results)

    system_prompt = """
You are ScholarRAG, an academic assistant that answers questions using
the user's uploaded textbook passages.

STRICT RULES:

1. Answer using ONLY the provided retrieved textbook context.
2. Do not invent facts or use outside knowledge.
3. If the context does not contain enough information, clearly say:
   "The retrieved textbook passages do not contain enough information
   to answer this confidently."
4. Give a direct answer to the question.
5. Explain concepts clearly and simply, like a helpful tutor.
6. Use bullet points when they improve clarity.
7. Do not dump entire retrieved passages.
8. Cite sources inline using this format:
   [filename, pages X-Y]
9. Cite the source supporting each important claim.
10. If multiple sources support the answer, cite them appropriately.

ANSWER FORMAT:

### Answer
A clear, concise answer to the question.

### Explanation
Explain the concept in understandable language when useful.

### Sources
List the textbook source(s) and page ranges used.

Remember: the textbook context is your only source of truth.
"""

    user_prompt = f"""
QUESTION:
{query}

RETRIEVED TEXTBOOK CONTEXT:
{context}

Write a helpful, accurate, textbook-grounded answer.
"""

    try:

        completion = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            temperature=0.1,
            max_completion_tokens=1200,
            messages=[
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ]
        )

        answer = completion.choices[0].message.content

        if not answer:
            return extractive_fallback(query, results)

        return answer

    except Exception as error:

        print("Groq generation error:", repr(error))

        return extractive_fallback(query, results)


print("ScholarRAG Groq answer generation is ready! 🚀")
print("Model:", GROQ_MODEL)

ScholarRAG Groq answer generation is ready! 🚀
Model: llama-3.3-70b-versatile


In [14]:
# ==========================================
# TEST SCHOLARRAG LLM ANSWER GENERATION
# ==========================================

test_question = "What are the main concepts discussed in this textbook?"

# Retrieve passages
test_candidates = hybrid_search(test_question, top_k=10)

# Rerank
test_reranked = rerank_results(
    test_question,
    test_candidates,
    top_k=5
)

# Generate AI answer
test_answer = generate_answer(
    test_question,
    test_reranked
)

print("=" * 80)
print("SCHOLARRAG AI ANSWER")
print("=" * 80)

print(test_answer)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Groq generation error: UnicodeEncodeError('ascii', 'Bearer # ========================================== # SCHOLARRAG — GROQ API KEY CONFIGURATION # ==========================================  import os from getpass import getpass  GROQ_API_KEY = getpass("Enter your Groq API key (hidden): ")  if not GROQ_API_KEY.strip():     raise ValueError("Groq API key was not provided.")  os.environ["GROQ_API_KEY"] = GROQ_API_KEY.strip()  print("Groq API key configured successfully! 🔐")', 65, 66, 'ordinal not in range(128)')
SCHOLARRAG AI ANSWER
⚠️ AI generation was unavailable. Here are the retrieved textbook passages:


### Source 1: textbook_full.pdf
**Pages:** 22-25

the state space and exchanging information between threads. Their main advantage is the use of crossovers — large blocks of letters that have evolved and lead to high valuations can be combined with other such blocks to produce a solution with a high total score. Figure 7: 8-Queens Solution 23 1.6 Summary 1.5 Summary We discussed se

In [15]:
import os

# Remove the incorrectly stored key
os.environ.pop("GROQ_API_KEY", None)

GROQ_API_KEY = ""

print("Incorrect Groq key cleared.")

Incorrect Groq key cleared.


In [17]:
from getpass import getpass
import os

key = getpass("Paste ONLY your Groq API key here: ").strip()

if not key or key.startswith("#") or " " in key:
    raise ValueError("That doesn't look like a valid API key input.")

os.environ["GROQ_API_KEY"] = key
GROQ_API_KEY = key

print("API key loaded successfully 🔐")
print("Key length:", len(key))
print("Key prefix:", key[:4] + "...")

Paste ONLY your Groq API key here: ··········
API key loaded successfully 🔐
Key length: 56
Key prefix: gsk_...


In [18]:
from groq import Groq
import os

client = Groq(api_key=os.environ["GROQ_API_KEY"])

try:
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "user", "content": "Reply with exactly: ScholarRAG connection successful!"}
        ],
        temperature=0,
        max_completion_tokens=30
    )

    print(response.choices[0].message.content)
    print("✅ Groq API connection successful!")

except Exception as e:
    print("❌ Groq connection failed:")
    print(type(e).__name__, str(e))

❌ Groq connection failed:
NotFoundError Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}


In [19]:
import os
from groq import Groq

# Use the API key already saved in this Colab runtime
api_key = os.environ.get("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY is missing. Set your API key first.")

# Initialize Groq client
client = Groq(api_key=api_key)

# Available model to test
GROQ_MODEL = "openai/gpt-oss-120b"

try:
    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "user",
                "content": "Reply with exactly: ScholarRAG connection successful!"
            }
        ],
        temperature=0,
        max_completion_tokens=50
    )

    print("Response:", response.choices[0].message.content)
    print("\n✅ GROQ CONNECTION SUCCESSFUL!")

except Exception as e:
    print("\n❌ GROQ CONNECTION FAILED")
    print("Error type:", type(e).__name__)
    print("Error details:", str(e))

Response: ScholarRAG connection successful!

✅ GROQ CONNECTION SUCCESSFUL!


In [20]:
# ============================================================
# SCHOLARRAG V2 — PHASE 6: GROUNDED LLM GENERATION
# ============================================================

import os
from groq import Groq

# ------------------------------------------------------------
# 1. GROQ CONFIGURATION
# ------------------------------------------------------------

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is missing. Please configure it first.")

GROQ_MODEL = "openai/gpt-oss-120b"

groq_client = Groq(api_key=GROQ_API_KEY)

print("✅ Groq client initialized")
print(f"🤖 Model: {GROQ_MODEL}")


# ------------------------------------------------------------
# 2. HELPER — SAFELY EXTRACT TEXT FROM RETRIEVED CHUNKS
# ------------------------------------------------------------

def extract_chunk_text(chunk):
    """Extract text from different possible chunk formats."""

    if isinstance(chunk, str):
        return chunk

    if isinstance(chunk, dict):
        possible_keys = [
            "text",
            "chunk_text",
            "content",
            "page_content",
            "document"
        ]

        for key in possible_keys:
            if key in chunk and chunk[key]:
                return str(chunk[key])

    # LangChain-style document
    if hasattr(chunk, "page_content"):
        return str(chunk.page_content)

    return str(chunk)


# ------------------------------------------------------------
# 3. HELPER — EXTRACT SOURCE INFORMATION
# ------------------------------------------------------------

def extract_chunk_metadata(chunk, index):
    """Extract source and page information from a chunk."""

    metadata = {}

    if isinstance(chunk, dict):
        metadata = chunk

    elif hasattr(chunk, "metadata"):
        metadata = getattr(chunk, "metadata", {}) or {}

    source = (
        metadata.get("source")
        or metadata.get("filename")
        or metadata.get("file_name")
        or metadata.get("document")
        or "Unknown source"
    )

    page_start = (
        metadata.get("page_start")
        or metadata.get("start_page")
        or metadata.get("page")
        or metadata.get("page_number")
    )

    page_end = (
        metadata.get("page_end")
        or metadata.get("end_page")
        or page_start
    )

    return {
        "source": source,
        "page_start": page_start,
        "page_end": page_end
    }


# ------------------------------------------------------------
# 4. BUILD CONTEXT FROM RERANKED CHUNKS
# ------------------------------------------------------------

def build_context(results):

    context_parts = []

    for i, chunk in enumerate(results, start=1):

        text = extract_chunk_text(chunk)
        metadata = extract_chunk_metadata(chunk, i)

        source = metadata["source"]
        page_start = metadata["page_start"]
        page_end = metadata["page_end"]

        if page_start is not None and page_end is not None:

            if page_start == page_end:
                page_info = f"Page {page_start}"
            else:
                page_info = f"Pages {page_start}-{page_end}"

        elif page_start is not None:
            page_info = f"Page {page_start}"

        else:
            page_info = "Page information unavailable"

        context_parts.append(
            f"""
[CONTEXT {i}]
Source: {source}
Location: {page_info}

{text}
"""
        )

    return "\n\n".join(context_parts)


# ------------------------------------------------------------
# 5. GROUNDED ANSWER GENERATION
# ------------------------------------------------------------

def generate_answer(query, results):

    # Handle empty retrieval
    if not results:
        return (
            "I couldn't find relevant information in the textbook "
            "to answer this question."
        )

    context = build_context(results)

    prompt = f"""
You are ScholarRAG, an AI assistant answering questions
using retrieved textbook information.

STRICT RULES:

1. Answer using ONLY the provided context.
2. Do not invent facts or information.
3. Do not use outside knowledge.
4. If the context does not contain enough information,
   explicitly say that the textbook context is insufficient.
5. Give a clear, accurate, educational answer.
6. Explain concepts in simple language when appropriate.
7. Use bullet points when they improve readability.
8. Include source citations using the provided source and page
   information.
9. Do not claim information is from the textbook unless it
   appears in the provided context.
10. If multiple retrieved passages are relevant, synthesize them.

QUESTION:

{query}


RETRIEVED TEXTBOOK CONTEXT:

{context}


ANSWER:
"""

    try:

        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,

            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are ScholarRAG, a grounded textbook "
                        "question-answering assistant. You must "
                        "strictly follow the provided context and "
                        "never fabricate textbook information."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],

            temperature=0.1,
            max_completion_tokens=1500
        )

        answer = response.choices[0].message.content

        if not answer:
            return "The model returned an empty answer."

        return answer.strip()

    except Exception as e:

        print("⚠️ Groq generation error:", type(e).__name__, str(e))

        # Fallback to retrieved text if generation fails
        fallback_parts = []

        for i, chunk in enumerate(results[:3], start=1):

            text = extract_chunk_text(chunk)
            metadata = extract_chunk_metadata(chunk, i)

            fallback_parts.append(
                f"Retrieved passage {i} "
                f"({metadata['source']}):\n{text}"
            )

        return (
            "⚠️ LLM generation failed. Here are the retrieved "
            "textbook passages instead:\n\n"
            + "\n\n".join(fallback_parts)
        )


print("✅ Phase 6: Grounded generation function ready!")
print(f"🤖 Using Groq model: {GROQ_MODEL}")

✅ Groq client initialized
🤖 Model: openai/gpt-oss-120b
✅ Phase 6: Grounded generation function ready!
🤖 Using Groq model: openai/gpt-oss-120b


In [21]:
# ============================================================
# SCHOLARRAG V2 — END-TO-END LLM TEST
# ============================================================

test_question = "What is artificial intelligence?"

print("🔍 Question:", test_question)
print("\n⏳ Retrieving relevant chunks...")

# Step 1: Hybrid retrieval
test_candidates = hybrid_search(
    test_question,
    top_k=10
)

print(f"✅ Retrieved {len(test_candidates)} candidate chunks")

# Step 2: Reranking
print("\n⏳ Reranking chunks...")

test_reranked = rerank_results(
    test_question,
    test_candidates,
    top_k=5
)

print(f"✅ Reranked to {len(test_reranked)} chunks")

# Step 3: Grounded LLM generation
print("\n⏳ Generating grounded answer using Groq...\n")

test_answer = generate_answer(
    test_question,
    test_reranked
)

# Step 4: Display answer
print("=" * 80)
print("📚 SCHOLARRAG V2 ANSWER")
print("=" * 80)

print(test_answer)

print("\n" + "=" * 80)
print("✅ END-TO-END TEST COMPLETED")
print("=" * 80)

🔍 Question: What is artificial intelligence?

⏳ Retrieving relevant chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Retrieved 10 candidate chunks

⏳ Reranking chunks...
✅ Reranked to 5 chunks

⏳ Generating grounded answer using Groq...

📚 SCHOLARRAG V2 ANSWER
**Artificial intelligence (AI)** is the study and engineering of *rational agents*—systems that have goals or preferences and choose actions that are expected to achieve the best possible outcomes given those goals.  In AI:

- An **agent** perceives its **environment** through sensors and influences it through actuators.  
- The central problem is to design agents that act **optimally** (i.e., rationally) in whatever world they inhabit.  
- AI focuses on the **core competencies** of intelligent systems:
  - **Problem solving** – finding sequences of actions that reach a goal.  
  - **Reasoning** – drawing logical conclusions from known facts.  
  - **Decision making** – selecting actions that maximize expected utility.  
  - **Learning** – improving performance from experience.  
- These competencies are built on **logical and probabilistic f

In [22]:
# ============================================================
# SCHOLARRAG V2 — HALLUCINATION CONTROL TEST
# ============================================================

hallucination_test_question = (
    "Explain the chemical process of photosynthesis "
    "in plants, including the role of chlorophyll."
)

print("🧪 HALLUCINATION CONTROL TEST")
print("=" * 80)
print("Question:", hallucination_test_question)

# Retrieve
hc_candidates = hybrid_search(
    hallucination_test_question,
    top_k=10
)

print(f"\nRetrieved chunks: {len(hc_candidates)}")

# Rerank
hc_reranked = rerank_results(
    hallucination_test_question,
    hc_candidates,
    top_k=5
)

print(f"Reranked chunks: {len(hc_reranked)}")

# Generate
hc_answer = generate_answer(
    hallucination_test_question,
    hc_reranked
)

print("\n" + "=" * 80)
print("SCHOLARRAG RESPONSE")
print("=" * 80)
print(hc_answer)

print("\n" + "=" * 80)
print("TEST COMPLETE")
print("=" * 80)

🧪 HALLUCINATION CONTROL TEST
Question: Explain the chemical process of photosynthesis in plants, including the role of chlorophyll.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Retrieved chunks: 10
Reranked chunks: 5

SCHOLARRAG RESPONSE
The textbook excerpts you provided discuss topics such as probabilistic inference, constraint‑satisfaction problems, and reinforcement‑learning algorithms. None of the retrieved passages contain any information about the chemical process of photosynthesis or the role of chlorophyll in plants.  

**Therefore, the textbook context is insufficient to answer the question about photosynthesis.** (Source: textbook_full.pdf, Pages 112‑115; 29‑31; 34‑35; 68‑70; 50‑52)

TEST COMPLETE


In [23]:
# ============================================================
# SCHOLARRAG V2 — FINAL 3-QUESTION ACCURACY TEST
# ============================================================

test_questions = [
    "What is artificial intelligence?",
    "What is a rational agent?",
    "What is the role of an agent's environment in artificial intelligence?"
]

for i, question in enumerate(test_questions, start=1):

    print("\n" + "#" * 90)
    print(f"TEST {i}/3")
    print("#" * 90)

    print(f"\n❓ QUESTION:\n{question}")

    # Retrieval
    candidates = hybrid_search(
        question,
        top_k=10
    )

    print(f"\n📚 Retrieved chunks: {len(candidates)}")

    # Reranking
    reranked = rerank_results(
        question,
        candidates,
        top_k=5
    )

    print(f"🎯 Reranked chunks: {len(reranked)}")

    # Generation
    answer = generate_answer(
        question,
        reranked
    )

    print("\n💡 ANSWER:")
    print(answer)

    print("\n" + "-" * 90)

print("\n✅ FINAL 3-QUESTION TEST COMPLETED")


##########################################################################################
TEST 1/3
##########################################################################################

❓ QUESTION:
What is artificial intelligence?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


📚 Retrieved chunks: 10
🎯 Reranked chunks: 5

💡 ANSWER:
**Artificial intelligence (AI)** is the study and engineering of **rational agents**—systems that have goals or preferences and choose actions that are expected to achieve the best possible outcomes for those goals.  

Key points from the textbook:

- **Core problem:** building an entity that can perceive its environment (via sensors), act on it (via actuators), and select actions that maximize its expected utility. 【Source: textbook_full.pdf, Pages 1‑3】
- **Core competencies of AI systems:**  
  - *Problem solving* – finding sequences of actions that lead to a goal.  
  - *Reasoning* – drawing logical conclusions from known facts.  
  - *Decision making* – choosing actions based on expected outcomes.  
  - *Learning* – improving performance from experience. 【Source: textbook_full.pdf, Pages 1‑3】
- **Foundations:** AI relies on logical and probabilistic methods to support these activities. 【Source: textbook_full.pdf, Pages 1‑3】

I

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


📚 Retrieved chunks: 10
🎯 Reranked chunks: 5

💡 ANSWER:
**Rational agent (as defined in the textbook)**  

A rational agent is an entity whose choices are governed by a set of well‑behaved preferences that allow its behavior to be represented as the maximization of expected utility.  Specifically, the agent’s preferences must satisfy the **five Axioms of Rationality**:

| Axiom | Meaning |
|-------|---------|
| **Orderability** | For any two outcomes A and B the agent either prefers A to B, prefers B to A, or is indifferent ( A ≻ B ∨ B ≻ A ∨ A ∼ B ). |
| **Transitivity** | If the agent prefers A to B and B to C, then it also prefers A to C ( A ≻ B ∧ B ≻ C ⇒ A ≻ C ). |
| **Continuity** | When A ≻ B ≻ C, there exists a probability p such that a lottery mixing A and C with probabilities p and 1‑p is indifferent to B. |
| **Substitutability** | If the agent is indifferent between A and B ( A ∼ B ), then it remains indifferent when A is substituted for B (or vice‑versa) in any lottery. |
| 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


📚 Retrieved chunks: 10
🎯 Reranked chunks: 5

💡 ANSWER:
**Role of the environment for an AI agent**

- **Defines the “world” in which the agent operates** – the environment is the setting that contains everything the agent can sense and act upon (e.g., the checkers board for a checkers‑playing agent)【Source: textbook_full.pdf, Pages 1‑3】.  
- **Provides percepts through sensors** – the agent receives information about the environment via its sensors, which it uses to decide what to do【Source: textbook_full.pdf, Pages 1‑3】.  
- **Receives the agent’s actions via actuators** – the agent influences the environment by executing actions with its actuators (e.g., moving a piece on the board)【Source: textbook_full.pdf, Pages 1‑3】.  
- **Shapes the design of the agent** – the type of environment (observable vs. partially observable, deterministic vs. stochastic, static vs. dynamic, single‑agent vs. multi‑agent, known vs. unknown physics) determines what kinds of reasoning, planning, or learnin

In [24]:
# ============================================================
# SCHOLARRAG V2 — FINAL GRADIO LAUNCH & VERIFICATION
# Includes all 4 steps:
# 1. Stop old Gradio interface
# 2. Launch updated Groq-powered interface
# 3. Generate and display Gradio link
# 4. Test 3 questions directly through the UI
# ============================================================

import gradio as gr
import time

print("=" * 70)
print("🚀 SCHOLARRAG V2 — FINAL UI LAUNCH")
print("=" * 70)


# ------------------------------------------------------------
# STEP 1 — STOP PREVIOUS GRADIO INTERFACE
# ------------------------------------------------------------

print("\n🛑 STEP 1: Stopping previous Gradio interface...")

# Try to close previously created Gradio interfaces
previous_interface_names = [
    "demo",
    "interface",
    "app",
    "gradio_app"
]

closed_previous = False

for name in previous_interface_names:

    old_interface = globals().get(name)

    if old_interface is not None:

        try:
            if hasattr(old_interface, "close"):
                old_interface.close()
                print(f"✅ Closed previous interface: {name}")
                closed_previous = True

        except Exception as e:
            print(f"⚠️ Could not close {name}: {e}")

if not closed_previous:
    print("ℹ️ No previous interface found to close.")

print("✅ Step 1 completed.")


# ------------------------------------------------------------
# STEP 2 — DEFINE THE RAG ANSWERING FUNCTION FOR GRADIO
# ------------------------------------------------------------

print("\n🔄 STEP 2: Preparing updated ScholarRAG pipeline...")


def scholarrag_ui_answer(question):

    question = str(question).strip()

    if not question:
        return "⚠️ Please enter a question."

    try:

        # ----------------------------------------
        # Hybrid Retrieval
        # ----------------------------------------

        candidates = hybrid_search(
            question,
            top_k=10
        )

        # ----------------------------------------
        # Cross-Encoder Reranking
        # ----------------------------------------

        reranked = rerank_results(
            question,
            candidates,
            top_k=5
        )

        # ----------------------------------------
        # Groq Grounded Generation
        # ----------------------------------------

        answer = generate_answer(
            question,
            reranked
        )

        # ----------------------------------------
        # Display pipeline information
        # ----------------------------------------

        output = f"""
📚 SCHOLARRAG V2 ANSWER

{answer}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔍 Pipeline Information

• Hybrid retrieved chunks: {len(candidates)}
• Reranked chunks: {len(reranked)}
• LLM: {GROQ_MODEL}
• Generation: Grounded with retrieved textbook context
"""

        return output

    except Exception as e:

        return (
            "❌ ScholarRAG encountered an error.\n\n"
            f"Error type: {type(e).__name__}\n"
            f"Error details: {str(e)}"
        )


print("✅ Updated Groq-powered answer function ready.")


# ------------------------------------------------------------
# STEP 3 — CREATE AND LAUNCH GRADIO UI
# ------------------------------------------------------------

print("\n🌐 STEP 3: Creating Gradio interface...")


TEST_QUESTIONS = [
    "What is artificial intelligence?",
    "What is a rational agent?",
    "Explain photosynthesis in plants."
]


with gr.Blocks(
    title="ScholarRAG V2",
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown(
        """
        # 📚 ScholarRAG V2

        ### Textbook Question Answering System

        Ask questions about your uploaded textbook.

        **Pipeline:** Hybrid Retrieval → Reranking → Groq LLM → Grounded Answer
        """
    )

    gr.Markdown("---")

    # Test question selector
    gr.Markdown("## 🧪 Built-in Test Questions")

    test_dropdown = gr.Dropdown(
        choices=TEST_QUESTIONS,
        value=TEST_QUESTIONS[0],
        label="Select a test question"
    )

    test_button = gr.Button(
        "🚀 Run Selected Test",
        variant="primary"
    )

    # Custom question input
    gr.Markdown("## 💬 Ask Your Own Question")

    question_input = gr.Textbox(
        label="Your question",
        placeholder="Ask a question about your textbook...",
        lines=3
    )

    submit_button = gr.Button(
        "🔍 Ask ScholarRAG",
        variant="primary"
    )

    # Answer output
    answer_output = gr.Textbox(
        label="📖 ScholarRAG Answer",
        lines=20,
        interactive=False
    )

    # ----------------------------------------
    # Test button
    # ----------------------------------------

    test_button.click(
        fn=scholarrag_ui_answer,
        inputs=test_dropdown,
        outputs=answer_output
    )

    # ----------------------------------------
    # Custom question button
    # ----------------------------------------

    submit_button.click(
        fn=scholarrag_ui_answer,
        inputs=question_input,
        outputs=answer_output
    )

    question_input.submit(
        fn=scholarrag_ui_answer,
        inputs=question_input,
        outputs=answer_output
    )


print("✅ Gradio interface created.")


# ------------------------------------------------------------
# LAUNCH INTERFACE
# ------------------------------------------------------------

print("\n🚀 Launching ScholarRAG V2...")

try:

    demo.launch(
        share=True,
        debug=False,
        show_error=True
    )

    print("\n" + "=" * 70)
    print("🎉 SCHOLARRAG V2 GRADIO INTERFACE LAUNCHED!")
    print("=" * 70)

    print("""
✅ STEP 1 — Previous interface stopped/closed
✅ STEP 2 — Updated Groq-powered pipeline loaded
✅ STEP 3 — Gradio interface launched
✅ STEP 4 — Three built-in test questions available

TEST QUESTIONS:

1. What is artificial intelligence?
2. What is a rational agent?
3. Explain photosynthesis in plants.

Use the dropdown and click 'Run Selected Test' to test them.
You can also type your own textbook questions.
""")

except Exception as e:

    print("\n❌ Gradio launch failed")
    print("Error type:", type(e).__name__)
    print("Error:", str(e))

🚀 SCHOLARRAG V2 — FINAL UI LAUNCH

🛑 STEP 1: Stopping previous Gradio interface...
Closing server running on port: 7860
✅ Closed previous interface: demo
✅ Step 1 completed.

🔄 STEP 2: Preparing updated ScholarRAG pipeline...
✅ Updated Groq-powered answer function ready.

🌐 STEP 3: Creating Gradio interface...


/tmp/ipykernel_548/294445830.py:146: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


✅ Gradio interface created.

🚀 Launching ScholarRAG V2...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://10be87eacb85c1d368.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



🎉 SCHOLARRAG V2 GRADIO INTERFACE LAUNCHED!

✅ STEP 1 — Previous interface stopped/closed
✅ STEP 2 — Updated Groq-powered pipeline loaded
✅ STEP 3 — Gradio interface launched
✅ STEP 4 — Three built-in test questions available

TEST QUESTIONS:

1. What is artificial intelligence?
2. What is a rational agent?
3. Explain photosynthesis in plants.

Use the dropdown and click 'Run Selected Test' to test them.
You can also type your own textbook questions.

